# Evaluación del modelo

Este informe evalúa los modelos de regresión entrenados en [Preparación del modelo](./preparacion_modelo.ipynb), con la misma definición del gasto futuro, las mismas cohortes y la misma prueba temporal. Se vuelven a ejecutar sus celdas para reconstruir las predicciones, evitando depender de objetos que existan solo en la memoria de otra sesión.

La variable objetivo es continua (gasto agregado futuro). Por ello se reportan métricas de **regresión**. Matriz de confusión, accuracy, precision, recall, F1-score y ROC-AUC son métricas de clasificación y **no corresponden** a este problema; no se discretiza el gasto con un umbral arbitrario.


In [1]:
from pathlib import Path

import nbformat
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

candidate_notebooks = [
    Path.cwd() / "preparacion_modelo.ipynb",
    Path.cwd() / "05_MACHINE_LEARNING" / "preparacion_modelo.ipynb",
    Path.cwd().parent / "05_MACHINE_LEARNING" / "preparacion_modelo.ipynb",
]
PREPARATION_NOTEBOOK = next(
    (path.resolve() for path in candidate_notebooks if path.exists()),
    None,
)
if PREPARATION_NOTEBOOK is None:
    raise FileNotFoundError(
        "No se encontró preparacion_modelo.ipynb. Ejecuta este notebook desde "
        "05_MACHINE_LEARNING o desde la raíz del proyecto."
    )

preparation = nbformat.read(PREPARATION_NOTEBOOK, as_version=4)
model_context = {}
for cell_number, cell in enumerate(preparation.cells):
    if cell.cell_type == "code":
        exec(
            compile(cell.source, f"{PREPARATION_NOTEBOOK.name}:celda-{cell_number}", "exec"),
            model_context,
        )

required_objects = {"y_test", "predictions", "selected_name", "metrics"}
missing_objects = required_objects.difference(model_context)
if missing_objects:
    raise RuntimeError(
        f"El notebook de preparación no produjo los objetos requeridos: {sorted(missing_objects)}"
    )

y_test = model_context["y_test"].astype(float)
predictions = model_context["predictions"]
print(f"Notebook evaluado: {PREPARATION_NOTEBOOK}")
print(f"Clientes en prueba temporal: {len(y_test):,}")


Archivo de entrada: C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA\03_ETL_DATA_QUALITY\online_retail_II_procesado.csv.gz
Directorio del proyecto: C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA
Dimensiones cargadas: 1,067,371 filas × 12 columnas
Rango de fechas: 2009-12-01 07:45:00 — 2011-12-09 12:50:00


,invoice,stock_code,quantity,invoice_date,price,customer_id,country,source_sheet,line_amount,is_cancellation,is_adjustment,is_exact_duplicate
0,489434,85048,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010,83.40,False,False,False
1,489434,79323P,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,81.00,False,False,False
2,489434,79323W,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010,81.00,False,False,False
3,489434,22041,48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010,100.80,False,False,False
4,489434,21232,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010,30.00,False,False,False


Filas originales leídas: 1,067,371
Filas excedentes por duplicado exacto retenidas fuera del modelado: 12,133
Filas después de deduplicar para este análisis: 1,055,238
Filas sin customer_id (no identificables por cliente): 243,007


,Conjunto,Clientes,Clientes con gasto futuro > 0,Gasto futuro total,Mediana del gasto futuro
0,Entrenamiento,2757,1932,"3,780,910.25",415.70
1,Prueba temporal,2724,1871,"3,832,576.06",390.83


Predictores numéricos: ['historical_spend', 'order_count', 'units_purchased', 'distinct_products', 'active_days', 'days_since_last_purchase', 'mean_order_spend', 'historical_return_lines', 'historical_cancellation_lines', 'historical_adjustment_lines']
Predictores categóricos: ['country']


,Modelo,MAE,RMSE,R²
0,Ridge,963.77,"4,928.33",0.48
1,Random Forest,"1,019.27","5,754.98",0.29
2,Baseline (mediana),"1,324.92","6,891.84",-0.02


Modelo seleccionado por menor MAE temporal: Ridge
Resumen de errores del modelo seleccionado, en la escala del gasto:


,count,mean,std,min,25%,50%,75%,max
Gasto real,"2,724.00","1,406.97","6,821.43",0.00,0.00,390.83,"1,150.80","168,469.60"
Gasto predicho,"2,724.00","1,314.21","3,679.40",1.85,259.61,591.33,"1,262.20","92,718.51"
Error absoluto,"2,724.00",963.77,"4,834.06",0.02,168.40,376.22,793.66,"168,442.06"


MAE = 963.77; RMSE = 4,928.33; R² = 0.478.
Notebook evaluado: C:\Users\USER\Documents\PROYECTO-FINAL-MINERIA\05_MACHINE_LEARNING\preparacion_modelo.ipynb
Clientes en prueba temporal: 2,724


## Métricas de regresión

- **MAE**: error absoluto medio, en las mismas unidades monetarias que el gasto. Resume el tamaño típico del error sin que unos pocos valores extremos dominen tanto el resultado.
- **MSE**: error cuadrático medio, en unidades monetarias al cuadrado. Penaliza con fuerza los errores grandes.
- **RMSE**: raíz del MSE, vuelve a las unidades del gasto y hace visible el efecto de errores extremos.
- **R²**: proporción de variabilidad del objetivo explicada respecto a predecir su media. No es un porcentaje de exactitud; puede ser negativo si el modelo es peor que esa referencia.

La comparación incluye la mediana como baseline, Ridge y Random Forest. Un menor MAE es el criterio de selección declarado en la preparación; RMSE y R² complementan la lectura.


In [2]:
metric_rows = []
for name, prediction in predictions.items():
    prediction = np.maximum(np.asarray(prediction, dtype=float), 0.0)
    metric_rows.append(
        {
            "Modelo": name,
            "MAE": mean_absolute_error(y_test, prediction),
            "MSE": mean_squared_error(y_test, prediction),
            "RMSE": np.sqrt(mean_squared_error(y_test, prediction)),
            "R²": r2_score(y_test, prediction),
        }
    )

evaluation_metrics = (
    pd.DataFrame(metric_rows)
    .sort_values(["MAE", "RMSE"])
    .reset_index(drop=True)
)
selected_name = evaluation_metrics.loc[0, "Modelo"]
baseline_row = evaluation_metrics.loc[
    evaluation_metrics["Modelo"].eq("Baseline (mediana)")
].iloc[0]
selected_row = evaluation_metrics.iloc[0]
mae_reduction_pct = 100 * (baseline_row["MAE"] - selected_row["MAE"]) / baseline_row["MAE"]
rmse_reduction_pct = 100 * (baseline_row["RMSE"] - selected_row["RMSE"]) / baseline_row["RMSE"]

display(evaluation_metrics)
print(f"Modelo seleccionado por menor MAE: {selected_name}")
print(
    f"Frente al baseline, el MAE se reduce {mae_reduction_pct:.1f}% "
    f"y el RMSE {rmse_reduction_pct:.1f}%."
)
assert selected_name == model_context["selected_name"], (
    "La selección no coincide con el criterio documentado en el notebook de preparación."
)


,Modelo,MAE,MSE,RMSE,R²
0,Ridge,963.77,"24,288,397.11","4,928.33",0.48
1,Random Forest,"1,019.27","33,119,743.71","5,754.98",0.29
2,Baseline (mediana),"1,324.92","47,497,449.17","6,891.84",-0.02


Modelo seleccionado por menor MAE: Ridge
Frente al baseline, el MAE se reduce 27.3% y el RMSE 28.5%.


## Tipos de error y lectura del modelo seleccionado

El error con signo se define como `predicción − valor real`: positivo indica sobreestimación y negativo, subestimación. Se separan además los clientes sin gasto futuro de aquellos con gasto positivo, porque los ceros y la cola de clientes de alto gasto tienen implicaciones distintas. Un promedio global puede ocultar esos errores por segmento.


In [3]:
selected_prediction = np.maximum(
    np.asarray(predictions[selected_name], dtype=float),
    0.0,
)
error_detail = pd.DataFrame(
    {
        "gasto_real": y_test.to_numpy(),
        "gasto_predicho": selected_prediction,
    }
)
error_detail["error"] = error_detail["gasto_predicho"] - error_detail["gasto_real"]
error_detail["error_absoluto"] = error_detail["error"].abs()
error_detail["segmento"] = np.where(
    error_detail["gasto_real"].eq(0),
    "Sin gasto futuro (y = 0)",
    "Con gasto futuro (y > 0)",
)

segment_rows = []
for segment, group in error_detail.groupby("segmento", sort=False):
    segment_rows.append(
        {
            "Segmento": segment,
            "Clientes": len(group),
            "MAE": group["error_absoluto"].mean(),
            "Error medio (predicción − real)": group["error"].mean(),
            "Gasto real medio": group["gasto_real"].mean(),
            "Gasto predicho medio": group["gasto_predicho"].mean(),
        }
    )
segment_metrics = pd.DataFrame(segment_rows)
display(segment_metrics)

overestimates = int(error_detail["error"].gt(0).sum())
underestimates = int(error_detail["error"].lt(0).sum())
exact_predictions = int(error_detail["error"].eq(0).sum())
print(
    f"Dirección del error: {overestimates:,} sobreestimaciones, "
    f"{underestimates:,} subestimaciones y {exact_predictions:,} predicciones exactas."
)


,Segmento,Clientes,MAE,Error medio (predicción − real),Gasto real medio,Gasto predicho medio
0,Con gasto futuro (y > 0),1871,"1,121.96",-416.25,"2,048.41","1,632.16"
1,Sin gasto futuro (y = 0),853,616.80,616.80,0.00,616.80


Dirección del error: 1,775 sobreestimaciones, 949 subestimaciones y 0 predicciones exactas.


## Informe y justificación de utilidad

La evaluación debe leerse en las unidades del campo `price`; el archivo no especifica explícitamente una divisa. El baseline sirve como referencia mínima: superar una predicción constante indica que las características históricas aportan información, pero no garantiza precisión suficiente para cada cliente.

La prueba temporal es posterior al entrenamiento, aunque se usó para comparar candidatos y seleccionar el menor MAE. Por eso las métricas son adecuadas para una comparación inicial, pero pueden ser optimistas como estimación final. Se recomienda una validación adicional en un periodo futuro aún no observado antes de fijar decisiones presupuestarias o comerciales.

### Limitaciones principales

- El objetivo es gasto transaccional elegible, no ventas netas contables ni margen; se excluyen cancelaciones, ajustes, devoluciones y líneas no positivas.
- Solo se modelan clientes identificados con compras elegibles históricas; no se evalúan clientes nuevos ni transacciones sin `customer_id`.
- Hay asimetría y valores extremos: RMSE puede ser mucho mayor que MAE, por lo que algunos clientes de alto gasto concentran errores relevantes.
- Una mejora sobre el baseline apoya su uso como señal de priorización exploratoria, no como pronóstico exacto por cliente. Antes de operarlo, comprobar ranking/top-decile, desempeño por país y estabilidad temporal, y acordar con negocio el costo de sobreestimar frente a subestimar.
- El periodo de prueba intervino en la elección entre modelos. Para una estimación imparcial posterior se necesita un conjunto de evaluación final que no intervenga en entrenamiento ni selección.


In [4]:
mae_value = selected_row["MAE"]
mse_value = selected_row["MSE"]
rmse_value = selected_row["RMSE"]
r2_value = selected_row["R²"]
mean_target = y_test.mean()
median_target = y_test.median()

print(
    f"Conclusión: {selected_name} es el mejor de los modelos comparados por MAE "
    f"({mae_value:,.2f}); RMSE = {rmse_value:,.2f}, MSE = {mse_value:,.2f} "
    f"y R² = {r2_value:.3f}. El gasto real medio es {mean_target:,.2f} "
    f"y la mediana {median_target:,.2f}. "
    f"Su ventaja de {mae_reduction_pct:.1f}% en MAE sobre el baseline justifica "
    "considerarlo para priorización exploratoria; el tamaño de los errores y la "
    "selección sobre la misma prueba impiden recomendarlo todavía para pronósticos "
    "financieros individuales sin validación adicional."
)


Conclusión: Ridge es el mejor de los modelos comparados por MAE (963.77); RMSE = 4,928.33, MSE = 24,288,397.11 y R² = 0.478. El gasto real medio es 1,406.97 y la mediana 390.83. Su ventaja de 27.3% en MAE sobre el baseline justifica considerarlo para priorización exploratoria; el tamaño de los errores y la selección sobre la misma prueba impiden recomendarlo todavía para pronósticos financieros individuales sin validación adicional.
